# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanhGiauTen/flyrankAI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
My lane is scoring. The model will assign each geographic observation an estimated median house value rather than place it into a fixed category. Scoring is appropriate because the target is continuous, and preserving the numerical differences between housing values provides more useful decision support than converting them into broad classes such as “low” or “high.”

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("/content/sample_data/california_housing_test.csv")

print("Target column:", "median_house_value")
print("Target data type:", df["median_house_value"].dtype)
print("Number of unique target values:", df["median_house_value"].nunique())
print("Target range:")
print(df["median_house_value"].agg(["min", "median", "max"]))

Target column: median_house_value
Target data type: float64
Number of unique target values: 1784
Target range:
min        22500.0
median    177650.0
max       500001.0
Name: median_house_value, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
The model would predict median_house_value, which is an observed aggregate outcome included in the dataset rather than a label created through a custom rule. It represents the recorded median house value for each geographic observation. However, it should be treated as a dataset-specific proxy for local housing value, not as the exact price of an individual property or a guaranteed estimate of its current market value.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target = "median_house_value"

print("Target:", target)
print("Missing target values:", df[target].isna().sum())
print("\nTarget summary:")
display(df[target].describe().to_frame())

Target: median_house_value
Missing target values: 0

Target summary:


,median_house_value
count,3000.00000
mean,205846.27500
std,113119.68747
min,22500.00000
25%,121200.00000
50%,177650.00000
75%,263975.00000
max,500001.00000


## 3. Success metric

*One metric you can defend. What number means 'good'?*
I will use Mean Absolute Error (MAE) because it measures the average absolute difference between estimated and observed median house values in the same unit as the target. A useful model should outperform a simple baseline that predicts the training-set median for every observation. For this initial project, I will consider the model good if its test MAE is at least 20% lower than the median-baseline MAE. This threshold represents a measurable improvement, although it does not guarantee that every individual estimate will be accurate.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Baseline: predict the training-set median for every test observation
baseline_prediction = y_train.median()
baseline_predictions = [baseline_prediction] * len(y_test)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
good_mae_threshold = baseline_mae * 0.80

print(f"Training-set median prediction: ${baseline_prediction:,.2f}")
print(f"Baseline test MAE: ${baseline_mae:,.2f}")
print(f"'Good' MAE threshold: below ${good_mae_threshold:,.2f}")


Training-set median prediction: $179,350.00
Baseline test MAE: $84,635.87
'Good' MAE threshold: below $67,708.70


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
One row represents one aggregated geographic housing observation in California. Each row contains observed demographic, geographic, and housing characteristics, including location, housing age, population, household count, income, and median house value. It does not represent one individual house, person, or transaction.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_columns = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "median_house_value"
]

lane_df = df[lane_columns].copy()

print(f"Unit of analysis: one aggregated geographic observation per row")
print(f"Number of rows: {len(lane_df):,}")
print(f"Number of columns: {lane_df.shape[1]}")
print(f"Duplicate rows: {lane_df.duplicated().sum()}")
print(f"Missing values: {lane_df.isna().sum().sum()}")

display(lane_df.head())
display(lane_df.iloc[[0]].T.rename(columns={0: "First observation"}))


Unit of analysis: one aggregated geographic observation per row
Number of rows: 3,000
Number of columns: 9
Duplicate rows: 0
Missing values: 0


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-122.05,37.37,27.0,3885.0,661.0,1537.0,606.0,6.6085,344700.0
1,-118.30,34.26,43.0,1510.0,310.0,809.0,277.0,3.5990,176500.0
2,-117.81,33.78,27.0,3589.0,507.0,1484.0,495.0,5.7934,270500.0
3,-118.36,33.82,28.0,67.0,15.0,49.0,11.0,6.1359,330000.0
4,-119.67,36.33,19.0,1241.0,244.0,850.0,237.0,2.9375,81700.0


,First observation
longitude,-122.0500
latitude,37.3700
housing_median_age,27.0000
total_rooms,3885.0000
total_bedrooms,661.0000
population,1537.0000
households,606.0000
median_income,6.6085
median_house_value,344700.0000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
A fixed rule is unlikely to represent this problem well because housing values may be associated with several interacting and potentially nonlinear factors. For example, the relevance of income may differ by location, while room counts must be interpreted alongside population and household counts. Geographic coordinates may also capture local patterns that cannot be represented adequately by a single threshold. Machine learning can estimate these multivariable patterns from observed data, but its output should remain decision support rather than causal proof or a guaranteed property valuation.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
numeric_correlations = (
    lane_df.corr(numeric_only=True)["median_house_value"]
    .drop("median_house_value")
    .sort_values(key=abs, ascending=False)
)

print("Observed correlations with median house value:")
display(numeric_correlations.to_frame("correlation"))


Observed correlations with median house value:


,correlation
median_income,0.672695
total_rooms,0.160427
latitude,-0.138428
households,0.100176
housing_median_age,0.091409
total_bedrooms,0.082279
longitude,-0.050662
population,-0.001192


In [6]:
pattern_check = lane_df.copy()

pattern_check["rooms_per_household"] = (
    pattern_check["total_rooms"] / pattern_check["households"]
)

pattern_check["bedrooms_per_room"] = (
    pattern_check["total_bedrooms"] / pattern_check["total_rooms"]
)

pattern_check["people_per_household"] = (
    pattern_check["population"] / pattern_check["households"]
)

selected_correlations = (
    pattern_check[
        [
            "median_income",
            "longitude",
            "latitude",
            "housing_median_age",
            "rooms_per_household",
            "bedrooms_per_room",
            "people_per_household",
            "median_house_value",
        ]
    ]
    .corr()["median_house_value"]
    .drop("median_house_value")
    .sort_values(key=abs, ascending=False)
)

display(selected_correlations.to_frame("observed_correlation"))

,observed_correlation
median_income,0.672695
bedrooms_per_room,-0.246313
rooms_per_household,0.153429
latitude,-0.138428
housing_median_age,0.091409
longitude,-0.050662
people_per_household,-0.045272


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.